# Data-driven GPU Underutilization Detection Framework

## Objective

This notebook identifies telemetry-defined GPU underutilization candidates.

It does not prove that a job was wasteful. Job-level average telemetry cannot
establish user intent, application state, or instantaneous GPU behaviour.

The framework:

1. isolates zero-compute allocations before statistical calibration;
2. calibrates low-compute behaviour from active workloads only;
3. estimates idle-power behaviour per `(Node, gpu_id)`;
4. treats I/O activity as workload context, not automatic waste;
5. reports measured energy associated with candidate categories.

In [91]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("1. Data/dcgm.csv")

dcgm_df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(dcgm_df):,}")
print(f"Columns: {len(dcgm_df.columns)}")
GROUP_COLUMNS = ["Node", "gpu_id"]

Rows: 96,893
Columns: 23


## 1. Distribution analysis

The full dataset is used to describe the cluster. Zero-SM observations are not
used to define the low-compute threshold because zero compute is a distinct
telemetry state, not merely the lower tail of active work.

In [92]:
required_columns = [
    "Node",
    "gpu_id",
    "smutilization_pct_avg",
    "memoryutilization_pct_avg",
    "pcierxbandwidth_megabytes_avg",
    "pcietxbandwidth_megabytes_avg",
    "powerusage_watts_avg",
    "energyconsumed_joules",
    "totalexecutiontime_sec",
]

missing_columns = set(required_columns) - set(dcgm_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = dcgm_df[required_columns].copy()

df["pcie_total_mb_s"] = (
    df["pcierxbandwidth_megabytes_avg"]
    + df["pcietxbandwidth_megabytes_avg"]
)

df["energy_kwh"] = df["energyconsumed_joules"] / 3_600_000

quality_checks = pd.Series({
    "rows": len(df),
    "duplicate_rows": df.duplicated().sum(),
    "missing_values": df.isna().sum().sum(),
    "negative_power": (df["powerusage_watts_avg"] < 0).sum(),
    "negative_energy": (df["energyconsumed_joules"] < 0).sum(),
    "invalid_sm_utilization": (
        (df["smutilization_pct_avg"] < 0)
        | (df["smutilization_pct_avg"] > 100)
    ).sum(),
    "invalid_memory_utilization": (
        (df["memoryutilization_pct_avg"] < 0)
        | (df["memoryutilization_pct_avg"] > 100)
    ).sum(),
})

display(quality_checks)


rows                          96893
duplicate_rows                  268
missing_values                    0
negative_power                    0
negative_energy                   0
invalid_sm_utilization            0
invalid_memory_utilization        0
dtype: int64

## 2. Zero-compute and idle-power reference

A job with average SM utilization equal to zero is placed in a separate
zero-compute category.

A quiet zero-compute reference population is used to estimate normal idle
power. Quiet means low memory utilization and low PCIe traffic relative to the
same `(Node, gpu_id)` group.

Power above the upper idle-power reference is an elevated-power signal. It is
not, by itself, proof of avoidable energy.

In [93]:
QUIET_QUANTILE = 0.25
IDLE_POWER_MAD_MULTIPLIER = 3
MIN_IDLE_REFERENCE_OBSERVATIONS = 30

df["zero_compute"] = df["smutilization_pct_avg"].eq(0)

# Quiet-state thresholds are calibrated per node/GPU group.
df["quiet_io_threshold_mb_s"] = (
    df.groupby(GROUP_COLUMNS)["pcie_total_mb_s"]
    .transform(lambda s: s.quantile(QUIET_QUANTILE))
)

df["quiet_memory_threshold_pct"] = (
    df.groupby(GROUP_COLUMNS)["memoryutilization_pct_avg"]
    .transform(lambda s: s.quantile(QUIET_QUANTILE))
)

df["quiet_io"] = (
    df["pcie_total_mb_s"] <= df["quiet_io_threshold_mb_s"]
)

df["quiet_memory"] = (
    df["memoryutilization_pct_avg"]
    <= df["quiet_memory_threshold_pct"]
)

df["idle_reference"] = (
    df["zero_compute"]
    & df["quiet_io"]
    & df["quiet_memory"]
    & (df["powerusage_watts_avg"] > 0)
)

idle_reference_df = df.loc[df["idle_reference"]].copy()

if idle_reference_df.empty:
    raise ValueError("No valid idle reference observations were found.")

def median_absolute_deviation(series):
    return (series - series.median()).abs().median()

global_idle_median = idle_reference_df["powerusage_watts_avg"].median()
global_idle_mad = median_absolute_deviation(
    idle_reference_df["powerusage_watts_avg"]
)

idle_baselines = (
    idle_reference_df
    .groupby(GROUP_COLUMNS)
    .agg(
        idle_reference_n=("powerusage_watts_avg", "size"),
        idle_power_median_w=("powerusage_watts_avg", "median"),
        idle_power_mad_w=("powerusage_watts_avg", median_absolute_deviation),
    )
    .reset_index()
)

df = df.merge(idle_baselines, on=GROUP_COLUMNS, how="left")

insufficient_reference = (
    df["idle_reference_n"].isna()
    | (df["idle_reference_n"] < MIN_IDLE_REFERENCE_OBSERVATIONS)
)

df.loc[insufficient_reference, "idle_power_median_w"] = global_idle_median
df.loc[insufficient_reference, "idle_power_mad_w"] = global_idle_mad

df["idle_power_upper_w"] = (
    df["idle_power_median_w"]
    + IDLE_POWER_MAD_MULTIPLIER * df["idle_power_mad_w"]
)

df["elevated_above_idle"] = (
    df["powerusage_watts_avg"] > df["idle_power_upper_w"]
)

print(f"Zero-compute observations: {df['zero_compute'].mean() * 100:.2f}%")
print(f"Global idle median: {global_idle_median:.2f} W")
print(f"Global idle MAD: {global_idle_mad:.2f} W")

Zero-compute observations: 38.95%
Global idle median: 26.57 W
Global idle MAD: 1.31 W


## 3. Low-compute threshold among active workloads

The low-compute threshold is calculated only from observations with non-zero
SM utilization. This avoids allowing the large zero-compute population to
collapse the active-workload threshold to zero.

In [94]:
LOW_COMPUTE_QUANTILE = 0.10

active_df = df.loc[
    df["smutilization_pct_avg"] > 0
].copy()

low_sm_threshold = active_df[
    "smutilization_pct_avg"
].quantile(LOW_COMPUTE_QUANTILE)

df["low_compute"] = (
    df["smutilization_pct_avg"].gt(0)
    & df["smutilization_pct_avg"].le(low_sm_threshold)
)

print(
    f"Low-compute threshold "
    f"(active-workload P10): {low_sm_threshold:.2f}%"
)
print(f"Low-compute observations: {df['low_compute'].mean() * 100:.2f}%")

Low-compute threshold (active-workload P10): 5.00%
Low-compute observations: 6.26%


## 4. Power interpretation

Power is evaluated relative to the local idle-power reference, not against a
single cluster-wide wattage threshold.

High power during active computation is not automatically anomalous. Elevated
power is relevant here only when paired with zero or low compute.

In [95]:
power_summary = df.groupby(
    ["zero_compute", "low_compute"]
)["powerusage_watts_avg"].describe()

display(power_summary)

print(
    "Observations above their local upper idle-power reference: "
    f"{df['elevated_above_idle'].mean() * 100:.2f}%"
)

count       mean        std     min        25%  \
zero_compute low_compute                                                     
False        False        53092.0  98.681448  48.839150   0.000  49.880900   
             True          6061.0  37.354069   5.261616  24.171  33.440400   
True         False        37740.0  27.063277   7.482219   0.000  25.320075   

                               50%         75%       max  
zero_compute low_compute                                  
False        False        96.60875  137.615250  242.4360  
             True         37.46910   40.468600   70.7525  
True         False        26.28900   28.357825  124.7010

Observations above their local upper idle-power reference: 68.59%


## 5. I/O context

High PCIe traffic is not classified as energy waste. A low-compute job with
high I/O may be staging data, communicating, or waiting on another system.

Without verified GPU model, PCIe generation, and link-width metadata, the
analysis can identify I/O activity relative to this cluster but cannot claim
physical PCIe saturation.

In [96]:
HIGH_IO_QUANTILE = 0.90
MIN_ACTIVE_GROUP_OBSERVATIONS = 30

global_high_io_threshold = active_df["pcie_total_mb_s"].quantile(
    HIGH_IO_QUANTILE
)

io_thresholds = (
    active_df
    .groupby(GROUP_COLUMNS)
    .agg(
        active_group_n=("pcie_total_mb_s", "size"),
        high_io_threshold_mb_s=(
            "pcie_total_mb_s",
            lambda s: s.quantile(HIGH_IO_QUANTILE)
        ),
    )
    .reset_index()
)

df = df.merge(io_thresholds, on=GROUP_COLUMNS, how="left")

insufficient_active_group = (
    df["active_group_n"].isna()
    | (df["active_group_n"] < MIN_ACTIVE_GROUP_OBSERVATIONS)
)

df.loc[
    insufficient_active_group,
    "high_io_threshold_mb_s"
] = global_high_io_threshold

df["high_io_relative"] = (
    df["pcie_total_mb_s"] > df["high_io_threshold_mb_s"]
)

df["io_bound_candidate"] = (
    df["low_compute"]
    & df["high_io_relative"]
)

print(
    "I/O-bound candidates: "
    f"{df['io_bound_candidate'].mean() * 100:.2f}%"
)

I/O-bound candidates: 0.88%


## 6. Long-running jobs

Duration is used to prioritize low-activity candidates, not as evidence of
waste by itself. A long active job may be completely legitimate.

In [97]:
LONG_DURATION_QUANTILE = 0.90

df["long_duration_threshold_sec"] = (
    df.groupby(GROUP_COLUMNS)["totalexecutiontime_sec"]
    .transform(lambda s: s.quantile(LONG_DURATION_QUANTILE))
)

df["long_running"] = (
    df["totalexecutiontime_sec"]
    > df["long_duration_threshold_sec"]
)

print(
    f"Long-running observations: "
    f"{df['long_running'].mean() * 100:.2f}%"
)

Long-running observations: 10.17%


## 7. Mutually exclusive candidate categories

The final assigned category is mutually exclusive. Individual telemetry
conditions may overlap, but the ordered category assignment reports each record
once, using the highest-priority applicable category. For
example, a long zero-compute job is reported as one long-running low-activity
candidate, not as several independent signals.

In [98]:
df["long_running_low_activity"] = (
    df["long_running"]
    & (df["zero_compute"] | df["low_compute"])
    & df["quiet_io"]
    & df["quiet_memory"]
    & df["elevated_above_idle"]
)

df["zero_compute_low_activity"] = (
    df["zero_compute"]
    & df["quiet_io"]
    & df["quiet_memory"]
    & df["elevated_above_idle"]
    & ~df["long_running_low_activity"]
)

df["low_compute_high_power"] = (
    df["low_compute"]
    & df["elevated_above_idle"]
    & ~df["high_io_relative"]
    & ~df["long_running_low_activity"]
)

df["zero_compute_with_activity"] = (
    df["zero_compute"]
    & (~df["quiet_io"] | ~df["quiet_memory"])
)

df["candidate_category"] = np.select(
    [
        df["long_running_low_activity"],
        df["zero_compute_low_activity"],
        df["io_bound_candidate"],
        df["low_compute_high_power"],
        df["zero_compute_with_activity"],
    ],
    [
        "Long-running low-activity candidate",
        "Zero-compute low-activity candidate",
        "I/O-bound candidate",
        "Low-compute high-power candidate",
        "Zero-compute with I/O or memory activity",
    ],
    default="Not flagged",
)

## 8. Energy associated with candidates

`energy_kwh` is measured energy reported by DCGM.

Results report measured energy associated with each telemetry-defined category.
They do not estimate recoverable or avoidable energy.

In [103]:


energy_risk_categories = [
    "Long-running low-activity candidate",
    "Zero-compute low-activity candidate",
    "Low-compute high-power candidate",
]

df["energy_risk_candidate"] = df["candidate_category"].isin(
    energy_risk_categories
)

summary = (
    df.groupby("candidate_category")
    .agg(
        jobs=("energy_kwh", "size"),
        measured_energy_kwh=("energy_kwh", "sum"),
    )
    .sort_values("measured_energy_kwh", ascending=False)
)

summary["share_of_jobs_pct"] = summary["jobs"] / len(df) * 100
summary["share_of_energy_pct"] = (
    summary["measured_energy_kwh"] / df["energy_kwh"].sum() * 100
)

display(summary.round(2))
total_cluster_energy_kwh = df["energy_kwh"].sum()

energy_risk_energy_kwh = df.loc[
    df["energy_risk_candidate"],
    "energy_kwh"
].sum()

energy_risk_share_pct = (
    energy_risk_energy_kwh / total_cluster_energy_kwh * 100
)

all_candidate_energy_kwh = df.loc[
    df["candidate_category"] != "Not flagged",
    "energy_kwh"
].sum()

all_candidate_share_pct = (
    all_candidate_energy_kwh / total_cluster_energy_kwh * 100
)

print(f"Total measured cluster energy: {total_cluster_energy_kwh:,.2f} kWh")

print(
    "Energy-risk candidate energy: "
    f"{energy_risk_energy_kwh:,.2f} kWh "
    f"({energy_risk_share_pct:.2f}% of total measured energy)"
)

print(
    "All candidate-category energy, including I/O-bound cases: "
    f"{all_candidate_energy_kwh:,.2f} kWh "
    f"({all_candidate_share_pct:.2f}% of total measured energy)"
)

,jobs,measured_energy_kwh,share_of_jobs_pct,share_of_energy_pct
candidate_category,,,,
Not flagged,62287,2327.64,64.28,87.59
Zero-compute with I/O or memory activity,27064,203.33,27.93,7.65
Low-compute high-power candidate,4539,75.27,4.68,2.83
Zero-compute low-activity candidate,1755,22.50,1.81,0.85
Long-running low-activity candidate,393,14.76,0.41,0.56
I/O-bound candidate,855,13.85,0.88,0.52


Total measured cluster energy: 2,657.35 kWh
Energy-risk candidate energy: 112.53 kWh (4.23% of total measured energy)
All candidate-category energy, including I/O-bound cases: 329.70 kWh (12.41% of total measured energy)


## 9. Simple sensitivity analysis

The main framework uses an elevated-power threshold of median idle power plus
3 MAD. This sensitivity analysis tests stricter and less strict versions of
that choice: 2 MAD, 3 MAD, and 4 MAD.

The low-compute threshold (active-workload P10) and quiet-state threshold (P25)
remain fixed. Therefore, this analysis evaluates sensitivity to the
idle-power threshold only.

In [105]:
sensitivity_results = []

for mad_multiplier in [2, 3, 4]:

    elevated_power = (
        df["powerusage_watts_avg"]
        > (
            df["idle_power_median_w"]
            + mad_multiplier * df["idle_power_mad_w"]
        )
    )

    long_running_low_activity = (
        df["long_running"]
        & (df["zero_compute"] | df["low_compute"])
        & df["quiet_io"]
        & df["quiet_memory"]
        & elevated_power
    )

    zero_compute_low_activity = (
        df["zero_compute"]
        & df["quiet_io"]
        & df["quiet_memory"]
        & elevated_power
        & ~long_running_low_activity
    )

    low_compute_high_power = (
        df["low_compute"]
        & elevated_power
        & ~df["high_io_relative"]
        & ~long_running_low_activity
    )

    candidate_mask = (
        long_running_low_activity
        | zero_compute_low_activity
        | low_compute_high_power
    )

    candidate_energy_kwh = df.loc[
        candidate_mask,
        "energy_kwh"
    ].sum()

    energy_share_pct = (
        candidate_energy_kwh
        / df["energy_kwh"].sum()
        * 100
    )

    sensitivity_results.append({
        "Idle-power threshold": f"Median + {mad_multiplier} MAD",
        "Candidate jobs": candidate_mask.sum(),
        "Candidate-job share (%)": candidate_mask.mean() * 100,
        "Associated measured energy (kWh)": candidate_energy_kwh,
        "Energy share of cluster (%)": energy_share_pct,
    })

sensitivity_table = pd.DataFrame(sensitivity_results)

display(sensitivity_table.round(2))

,Idle-power threshold,Candidate jobs,Candidate-job share (%),Associated measured energy (kWh),Energy share of cluster (%)
0,Median + 2 MAD,7183,7.41,114.78,4.32
1,Median + 3 MAD,6687,6.90,112.53,4.23
2,Median + 4 MAD,6238,6.44,110.99,4.18
